# Notebook 1 — Earnings Event Study (any tickers) — v3
Method: **FIN 559 Lecture 15** (CAPM event study) + **Lecture 16** (CAR windows, t-tests).
v3 fixes the lxml failure: setup now installs `lxml`/`html5lib` (older yfinance scrapes Yahoo's earnings page as HTML), and if both live sources still fail, the notebook falls back to a local `earnings_events.csv` in this folder (columns: ticker, report_date, eps_est, eps_act) so it always runs. Nothing is skipped silently.

In [ ]:
# Setup: installs deps into THIS kernel if missing (FIN 559 Lecture 3 pattern)
# lxml/html5lib are REQUIRED by older yfinance versions: they scrape Yahoo's earnings
# page as HTML and die with "Missing optional dependency 'lxml'" without them.
import sys, importlib
for pkg in ["yfinance", "scipy", "statsmodels", "lxml", "html5lib"]:
    try: importlib.import_module(pkg)
    except ImportError: get_ipython().system(f"{sys.executable} -m pip install -q {pkg}")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, yfinance as yf
from scipy import stats
print("ready")

## Config — edit this cell only

In [ ]:
TICKERS = ["GOOGL","MSFT","AMZN","AAPL","PLTR","XOM","CVX","META","EQT","INTC","AMD","SWK","PBF","LLY"]
N_QUARTERS = 12
MARKET = "SPY"
EST_WIN = (-70, -11)
FALL_MONTHS = (10, 11)
COLS = ["ticker","report_date","fall","surprise_pct","beat","miss","react_ar_capm","react_ar_madj","car_p1_p3","car_p1_p5","car_m5_p5"]

## Build the event panel

In [ ]:
px_all = yf.download(TICKERS + [MARKET], period="5y", auto_adjust=False, progress=False)["Close"]
px_all.index = px_all.index.tz_localize(None).normalize()
mkt_ret = px_all[MARKET].pct_change()

def get_ed_local(tk):
    """Fallback 3: reuse a local earnings_events.csv (ticker, report_date, eps_est, eps_act)."""
    try:
        df = pd.read_csv("earnings_events.csv"); df = df[df.ticker == tk]
        if not len(df): return None
        df.index = pd.to_datetime(df.report_date)
        return df.rename(columns={"eps_est": "est", "eps_act": "act"})[["est", "act"]].head(N_QUARTERS)
    except Exception: return None

def get_ed(t, tk):
    try:
        ed = t.get_earnings_dates(limit=N_QUARTERS + 4)
    except Exception as e:
        print("  get_earnings_dates failed:", e); ed = None
    if ed is None or len(ed) == 0:
        try: ed = t.earnings_dates
        except Exception: ed = None
    if ed is None or len(ed) == 0:
        ed = get_ed_local(tk)
        if ed is not None: print("  using local earnings_events.csv fallback")
    if ed is None or len(ed) == 0: return None
    ed = ed.rename(columns={"EPS Estimate": "est", "Reported EPS": "act"})
    if "est" not in ed or "act" not in ed: return None
    ed.index = pd.to_datetime(ed.index).tz_localize(None)
    return ed[ed["act"].notna()].head(N_QUARTERS)

def reaction_idx(days, when):
    ann = when.normalize()
    if when.hour < 9 or (when.hour == 9 and when.minute < 30):
        cand = days[days == ann]
        if len(cand): return days.get_loc(cand[0])
    after = days[days > ann]
    return days.get_loc(after[0]) if len(after) else None

rows = []
for tk in TICKERS:
    try:
        ed = get_ed(yf.Ticker(tk), tk)
        if ed is None: print(f"{tk}: NO earnings dates from any source"); continue
        px = px_all[tk].dropna(); ret, days = px.pct_change(), px.index
        built = 0
        for when, r in ed.iterrows():
            est, act = r["est"], r["act"]
            if pd.isna(est) or pd.isna(act) or est == 0: continue
            ri = reaction_idx(days, when)
            if ri is None or ri + EST_WIN[0] < 1 or ri + 6 >= len(days): continue
            X = mkt_ret.reindex(days[ri+EST_WIN[0]: ri+EST_WIN[1]+1]); Y = ret.iloc[ri+EST_WIN[0]: ri+EST_WIN[1]+1]
            j = pd.concat([Y, X], axis=1).dropna()
            if len(j) < 40: continue
            slope, intercept = np.polyfit(j.iloc[:,1], j.iloc[:,0], 1)
            def ar_capm(i): return ret.iloc[i] - intercept - slope * mkt_ret.get(days[i], np.nan)
            def ar_madj(i): return ret.iloc[i] - mkt_ret.get(days[i], np.nan)
            win_c = {d: ar_capm(ri+d) for d in range(-5,6)}; win_m = {d: ar_madj(ri+d) for d in range(-5,6)}
            if any(pd.isna(v) for v in win_c.values()): continue
            surp = (act-est)/abs(est)
            rows.append(dict(ticker=tk, report_date=str(when.normalize().date()), fall=when.month in FALL_MONTHS,
                surprise_pct=round(100*surp,1), beat=surp>0.01, miss=surp<-0.01,
                react_ar_capm=round(100*win_c[0],2), react_ar_madj=round(100*win_m[0],2),
                car_p1_p3=round(100*sum(win_c[d] for d in range(1,4)),2),
                car_p1_p5=round(100*sum(win_c[d] for d in range(1,6)),2),
                car_m5_p5=round(100*sum(win_c[d] for d in range(-5,6)),2)))
            built += 1
        print(f"{tk}: {built} events built from {len(ed)} reported quarters")
    except Exception as e:
        print(f"{tk}: FAILED with {type(e).__name__}: {e}")
ev = pd.DataFrame(rows, columns=COLS)
print(f"TOTAL: {len(ev)} events")
if len(ev) == 0: print("Panel is EMPTY - read the per-ticker prints above; do not run the cells below.")
else: ev.to_csv("earnings_events_capm.csv", index=False); ev.head()

## Beat vs miss — Lecture 15's two-sample test + Lecture 16's one-sample CAR tests

In [ ]:
if len(ev):
    for lbl, g in [("BEAT", ev[ev.beat]), ("MISS", ev[ev.miss])]:
        if len(g) >= 3:
            t1, p1 = stats.ttest_1samp(g.react_ar_capm, 0)
            print(f"{lbl:5} n={len(g):3}  CAPM reaction {g.react_ar_capm.mean():+6.2f}% (t={t1:.2f}, p={p1:.3f})  "
                  f"drift+1..5 {g.car_p1_p5.mean():+5.2f}%  hit {(g.react_ar_capm>0).mean()*100:.0f}%")
    b, m = ev[ev.beat].react_ar_capm, ev[ev.miss].react_ar_capm
    if len(b) >= 3 and len(m) >= 3:
        print("Levene (variance) p:", round(stats.levene(b, m).pvalue, 3),
              "| two-sample t (beat vs miss) p:", round(stats.ttest_ind(b, m).pvalue, 4))

## Per-ticker + fall-print (this-season) view

In [ ]:
if len(ev):
    summ = ev.groupby("ticker").agg(n=("react_ar_capm","size"), beat_rate=("beat","mean"),
            mean_react=("react_ar_capm","mean"), mean_abs=("react_ar_capm", lambda s: s.abs().mean()),
            drift5=("car_p1_p5","mean")).round(2); summ["beat_rate"] *= 100; print(summ)
    f = ev[ev.fall].groupby("ticker").agg(fall_n=("react_ar_capm","size"), fall_react=("react_ar_capm","mean"),
            fall_drift5=("car_p1_p5","mean")).round(2); print(f[f.fall_n >= 2])

## Visual — earnings reactions at a glance
Two pictures of the same table: **every print as a dot** (is the reaction about direction or just size?), and **mean reaction vs average |reaction|** with the fall prints dotted in black — the fall dots are the only ones describing THIS season.

In [ ]:
# VISUAL A - every print as a dot: reaction-day CAPM abnormal return, green = beat, red = miss, black ring = fall print
import sys, importlib
try: import matplotlib.pyplot as plt
except ImportError: get_ipython().system(f"{sys.executable} -m pip install -q matplotlib"); import matplotlib.pyplot as plt
if len(ev):
    order = ev.groupby("ticker").react_ar_capm.mean().sort_values().index.tolist()
    fig, ax = plt.subplots(figsize=(12, 5))
    for j, tk in enumerate(order):
        g = ev[ev.ticker == tk]
        ax.scatter([j]*len(g), g.react_ar_capm, c=["green" if b else "red" for b in g.beat], s=45, alpha=0.75)
        f = g[g.fall]
        ax.scatter([j]*len(f), f.react_ar_capm, s=170, facecolors="none", edgecolors="black", linewidths=1.2)
    ax.axhline(0, color="black", lw=0.9)
    ax.set_xticks(range(len(order))); ax.set_xticklabels(order)
    ax.set_ylabel("reaction-day abnormal return % (CAPM)")
    ax.set_title("Every earnings reaction: green = beat, red = miss, black ring = fall print (this season)")
    ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# VISUAL B - mean reaction (colored bars) vs average |reaction| (grey) with fall-print means as black dots
import sys, importlib
try: import matplotlib.pyplot as plt
except ImportError: get_ipython().system(f"{sys.executable} -m pip install -q matplotlib"); import matplotlib.pyplot as plt
if len(ev):
    s = ev.groupby("ticker").agg(mr=("react_ar_capm", "mean"), ma=("react_ar_capm", lambda x: x.abs().mean()))
    fl = ev[ev.fall].groupby("ticker").react_ar_capm.mean()
    s = s.sort_values("mr")
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(s.index, s.ma, color="lightgrey", label="avg |reaction| (how much it moves)")
    ax.barh(s.index, s.mr, color=["green" if v > 0 else "red" for v in s.mr], label="mean reaction (which way)")
    ax.scatter(fl.values, fl.index, color="black", zorder=5, label="fall-print mean reaction")
    ax.axvline(0, color="black", lw=0.9); ax.set_xlabel("%")
    ax.set_title("Reaction size vs direction by ticker - black dots are the prints that matter this season")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()